# YOLO11 图像分类应用案例

本案例基于 MindSpore 适配实现的 `ultralytics`，演示 YOLO11 图像分类任务的训练、评估与推理完整流程。

In [18]:
import importlib
import sys
from pathlib import Path

import mindspore as ms

custom_ultralytics_parent = Path("/root/mindnlp/src/mindnlp")
if str(custom_ultralytics_parent) not in sys.path:
    sys.path.insert(0, str(custom_ultralytics_parent))

# 清理所有已加载的 ultralytics 缓存，避免混入官方 PyTorch 版
for module_name in list(sys.modules):
    if module_name == "ultralytics" or module_name.startswith("ultralytics."):
        sys.modules.pop(module_name, None)

importlib.invalidate_caches()

import ultralytics
from ultralytics import YOLO

ms.set_context(mode=ms.PYNATIVE_MODE, device_target="Ascend")

package_dir = Path(ultralytics.__file__).resolve().parent
work_dir = Path.cwd() / "demo_inputs"
work_dir.mkdir(parents=True, exist_ok=True)

print("MindSpore version:", ms.__version__)
print("ultralytics package:", ultralytics.__file__)
print("package_dir:", package_dir)

[WARNING] ME(31722:281473491372608,MainProcess):2026-04-24-17:47:15.121.000 [mindspore/context.py:1334] For 'context.set_context', the parameter 'device_target' will be deprecated and removed in a future version. Please use the api mindspore.set_device() instead.


MindSpore version: 2.8.0
ultralytics package: /root/mindnlp/src/mindnlp/ultralytics/__init__.py
package_dir: /root/mindnlp/src/mindnlp/ultralytics


## 1. 数据与测试图片准备

训练与评估默认使用 `cfg/datasets/imagenette2-160.yaml`。推理阶段优先使用分类数据集中的样例图片；若当前环境下没有可用测试图片，则自动下载一张示例图片。

In [19]:
data_yaml = package_dir / "cfg/datasets/imagenette2-160.yaml"
finetune_model_path = package_dir / "yolo11n-cls.pt"
scratch_model_path = package_dir / "cfg/models/11/yolo11-cls.yaml"

def resolve_source():
    candidates = [
        package_dir / "datasets/imagenette2-160/val",
        package_dir / "datasets/imagenette2-160/train",
    ]
    suffixes = {".jpg", ".jpeg", ".png", ".bmp"}
    for candidate in candidates:
        if candidate.is_file():
            return candidate
        if candidate.is_dir():
            files = sorted([p for p in candidate.rglob("*") if p.suffix.lower() in suffixes])
            if files:
                return files[0]

    image_path = work_dir / "classify_demo.jpg"
    if not image_path.exists():
        urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", image_path.as_posix())
        print("测试图片下载完成:", image_path)
    else:
        print("测试图片已存在:", image_path)
    return image_path

source_img = resolve_source()
print("数据配置:", data_yaml)
print("推理图片:", source_img)

数据配置: /root/mindnlp/src/mindnlp/ultralytics/cfg/datasets/imagenette2-160.yaml
推理图片: /root/mindnlp/src/mindnlp/ultralytics/datasets/imagenette2-160/val/n01440764/ILSVRC2012_val_00009111.JPEG


## 2. 模型训练

分类任务支持使用预训练权重进行微调，也支持使用模型配置文件从头开始训练。

In [25]:
import os
os.chdir(package_dir)
print("当前工作目录:", Path.cwd())

# 微调
#model = YOLO(finetune_model_path.as_posix())
# 从头开始训练
model = YOLO(scratch_model_path.as_posix())

train_results = model.train(
    data=data_yaml.as_posix(),
    epochs=100,
    imgsz=224,
    batch=64,
    amp=False,
    val_interval=1,
    workers=8
)

print("训练完成。")
print("best_fitness:", getattr(train_results, "best_fitness", None))
print("save_dir:", getattr(train_results, "save_dir", None))

当前工作目录: /root/mindnlp/src/mindnlp/ultralytics
[MindNLP YOLO] 检测到传入 YAML 架构文件: /root/mindnlp/src/mindnlp/ultralytics/cfg/models/11/yolo11-cls.yaml
[MindNLP YOLO] 模式: 从头开始随机初始化训练 (跳过权重转换)。
[MindNLP YOLO] 准备启动 classify 任务的训练...
[INFO] 训练任务启动，总轮数: 1 epochs
Epoch [0/0] Step [0/147] | Loss: 2.3760
Epoch [0/0] Step [10/147] | Loss: 2.3020
Epoch [0/0] Step [20/147] | Loss: 2.3034
Epoch [0/0] Step [30/147] | Loss: 2.3253
Epoch [0/0] Step [40/147] | Loss: 2.3476
Epoch [0/0] Step [50/147] | Loss: 2.2402
Epoch [0/0] Step [60/147] | Loss: 2.2737
Epoch [0/0] Step [70/147] | Loss: 2.3028
Epoch [0/0] Step [80/147] | Loss: 2.2017
Epoch [0/0] Step [90/147] | Loss: 2.1495
Epoch [0/0] Step [100/147] | Loss: 2.0472
Epoch [0/0] Step [110/147] | Loss: 2.0304
Epoch [0/0] Step [120/147] | Loss: 2.0184
Epoch [0/0] Step [130/147] | Loss: 1.9591
Epoch [0/0] Step [140/147] | Loss: 1.9182

[INFO] 开始执行 Epoch 0 验证程序...


Validating: 100%|██████████| 62/62 [00:38<00:00,  1.59it/s]

推理测速: preprocess: 0.0ms | inference: 8.7ms | postprocess: 0.6ms
验证结果 | Top-1 Acc: 0.0991 | Top-5 Acc: 0.5208


--------------------------------------------------
[评估报告] Epoch 0
  - accuracy_top1   : 0.09911
  - accuracy_top5   : 0.52076
[INFO] 当前模型综合评价指标 (Fitness): 0.30994
--------------------------------------------------

[INFO] 已更新最佳模型权重 (best.ckpt)，当前最高精度: 0.3099
训练完成。
best_fitness: 0.30993630573248404
save_dir: runs/classify/train


## 3. 模型评估

训练完成后，在验证集上执行图像分类评估。

In [26]:
val_results = model.val(data=data_yaml.as_posix())
print("评估完成。")
print(val_results)

[MindNLP YOLO] 准备启动 classify 任务的验证...


Validating: 100%|██████████| 246/246 [00:53<00:00,  4.56it/s]

推理测速: preprocess: 0.0ms | inference: 12.6ms | postprocess: 0.5ms
验证结果 | Top-1 Acc: 0.0991 | Top-5 Acc: 0.5195


评估完成。
{'speed': {'preprocess': 0.0031651053459021694, 'inference': 12.608467089902064, 'postprocess': 0.4601196118980456}, 'metrics/accuracy_top1': 0.09910828025477707, 'metrics/accuracy_top5': 0.5194904458598726, 'fitness': 0.3092993630573248}


## 4. 模型推理

推理结果会保存到输出目录，用户可以根据目录中的结果图片或日志查看预测结果。

In [27]:

predict_results = model(
    source=source_img.as_posix(),
    imgsz=224,
    save=True,
)

print("推理完成。")
if len(predict_results) > 0:
    print("推理结果保存目录:", getattr(predict_results[0], "save_dir", None))

[MindNLP YOLO] 准备启动 classify 任务的推理...
 推理结果将保存至: /root/mindnlp/src/mindnlp/ultralytics/runs/detect/predict
推理引擎启动，共探测到 1 份输入样本。
处理完成 [ILSVRC2012_val_00009111.JPEG] | 前向推理: 64.0ms | 后处理: 34.4ms
推理完成。
推理结果保存目录: /root/mindnlp/src/mindnlp/ultralytics/runs/detect/predict
